### 1. Importaciones y configuración global
En esta celda importamos todas las librerías necesarias para el manejo de datos, visualización y creación de redes neuronales (PyTorch y PyTorch Geometric). También configuramos la semilla de reproducibilidad y definimos si el código se ejecutará en la CPU o en la tarjeta gráfica (GPU).

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, roc_curve, accuracy_score,
                             precision_score, recall_score, f1_score,
                             confusion_matrix, average_precision_score)

from torch_geometric.data import Data, DataLoader as GeoDataLoader
from torch_geometric.nn import EdgeConv, global_mean_pool, global_max_pool
from torch_geometric.utils import to_dense_adj

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositiu: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### 2. Nomenclatura de las variables
Aquí definimos los nombres de las columnas que componen nuestro dataset. Se separan en variables de bajo nivel (características físicas directas de las partículas) y variables de alto nivel (cantidades derivadas). También identificamos los índices de las variables de momento transversal (pT) de los jets para un filtrado posterior.

In [ ]:
LOW_LEVEL_COLS = [
    'lepton_pT', 'lepton_eta', 'lepton_phi',
    'missing_energy_magnitude', 'missing_energy_phi',
    'jet1_pt', 'jet1_eta', 'jet1_phi', 'jet1_b_tag',
    'jet2_pt', 'jet2_eta', 'jet2_phi', 'jet2_b_tag',
    'jet3_pt', 'jet3_eta', 'jet3_phi', 'jet3_b_tag',
    'jet4_pt', 'jet4_eta', 'jet4_phi', 'jet4_b_tag',
]
HIGH_LEVEL_COLS = [
    'm_jj', 'm_jjj', 'm_lv', 'm_jlv', 'm_bb', 'm_wbb', 'm_wwbb'
]
ALL_COLS = ['label'] + LOW_LEVEL_COLS + HIGH_LEVEL_COLS

JET_PT_COLS = ['jet1_pt', 'jet2_pt', 'jet3_pt', 'jet4_pt']
JET_PT_IDX  = [LOW_LEVEL_COLS.index(c) for c in JET_PT_COLS]

### 3. Carga de los datos
En este paso leemos una porción del conjunto de datos masivo (HIGGS.csv) desde nuestro almacenamiento local. Extraemos el número especificado de muestras y comprobamos el tamaño resultante y la proporción entre eventos de señal y eventos de ruido de fondo.

In [ ]:
N_SAMPLES = 6_000_000

print(f"Carregant {N_SAMPLES:,} mostres del dataset HIGGS...")
t0 = time.time()
df = pd.read_csv("HIGGS.csv", header=None, nrows=N_SAMPLES, names=ALL_COLS)
print(f"  Carregat en {time.time()-t0:.1f}s  |  Shape: {df.shape}")
print(f"  Balanç: {df['label'].mean()*100:.1f}% senyal, "
      f"{(1-df['label'].mean())*100:.1f}% fons")

### 4. Análisis Exploratorio (EDA)
Visualizamos cómo se distribuyen algunas de las variables físicas comparando los casos de "señal" frente a los de "fondo". Además, calculamos la asimetría estadística de los datos y generamos un mapa de calor que muestra qué tan correlacionadas están las distintas características.

In [ ]:
os.makedirs('figures', exist_ok=True)

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

colors = {'senyal': '#2196F3', 'fons': '#F44336'}
signal     = df[df['label'] == 1]
background = df[df['label'] == 0]

plot_vars = LOW_LEVEL_COLS[:12]
for ax, col in zip(axes, plot_vars):
    ax.hist(background[col].sample(50_000, replace=True), bins=60, alpha=0.55,
            color=colors['fons'], label='Fons', density=True)
    ax.hist(signal[col].sample(50_000, replace=True), bins=60, alpha=0.55,
            color=colors['senyal'], label='Senyal', density=True)
    ax.set_title(col.replace('_', ' '), fontsize=9)
    ax.set_xlabel('Valor', fontsize=8)
    ax.tick_params(labelsize=7)
    if col == plot_vars[0]:
        ax.legend(fontsize=8)

plt.suptitle('Distribucions de les 12 primeres variables de baix nivell\n'
             '(Blau: senyal | Vermell: fons)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('figures/distributions.pdf', bbox_inches='tight', dpi=150)

skewness = df[LOW_LEVEL_COLS].skew().sort_values(ascending=False)
print("\nAssimetria per variable:")
print(skewness.round(3))

fig, ax = plt.subplots(figsize=(14, 12))
corr = df[LOW_LEVEL_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            annot=False, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Matriu de correlació — variables de baix nivell', fontsize=13)
plt.tight_layout()
plt.savefig('figures/correlation_matrix.pdf', bbox_inches='tight', dpi=150)

### 5. Preprocesamiento
Preparamos los datos para las redes neuronales. Los dividimos en entrenamiento, validación y prueba. Aplicamos la transformación matemática de Yeo-Johnson para corregir la asimetría en los valores y, posteriormente, los estandarizamos para que todos operen en la misma escala numérica.

In [ ]:
X = df[LOW_LEVEL_COLS].values.astype(np.float32)
y = df['label'].values.astype(np.float32)

jet_pt_raw = X[:, JET_PT_IDX].copy()

X_train, X_temp, y_train, y_temp, pt_train, pt_temp = train_test_split(
    X, y, jet_pt_raw, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test, pt_val, pt_test = train_test_split(